# ASL Citizen Full Dataset Training (Interactive)

This notebook provides an interactive interface for training and analyzing the LSTM model on ASL Citizen dataset.

**Note**: For long-running training (100 epochs, 48+ hours), use the Python script:
```bash
python ../scripts/train_asl_lstm.py --epochs 100
```

This notebook is best for:
- Quick experimentation (5-20 epochs)
- Visualization and analysis
- Understanding the model
- Testing on subsets

## Dataset
- **ASL Citizen**: 83,402 videos, 2,731 sign classes
- **Features**: MediaPipe Hands (42 keypoints × 3 coords = 126 features)
- **Model**: Bidirectional LSTM
- **Framework**: PyTorch

## 1. Setup and Imports

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from tqdm.notebook import tqdm
from sklearn.metrics import confusion_matrix, classification_report

# Resolve paths relative to this notebook's location
NOTEBOOK_DIR = Path(os.path.dirname(os.path.abspath("__file__"))).resolve()
ML_DIR = NOTEBOOK_DIR.parent

# Add scripts to path using absolute resolved path
sys.path.insert(0, str(ML_DIR / 'scripts'))
from asl_dataset import ASLCitizenDataset, create_data_loaders

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seeds
import random
random.seed(42)
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 2. Configuration

In [ ]:
# Paths — using absolute resolved paths from ML_DIR (set in cell 2)
DATA_DIR = ML_DIR / 'data'
CSV_DIR = DATA_DIR / 'data_csv'
POSE_DIR = DATA_DIR / 'processed' / 'pose_per_files'
SAVE_DIR = ML_DIR / 'trained_models'
LOG_DIR = ML_DIR / 'logs'

# Ensure output directories exist
SAVE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

train_csv = CSV_DIR / 'train.csv'
val_csv = CSV_DIR / 'val.csv'
test_csv = CSV_DIR / 'test.csv'

pose_map_train = CSV_DIR / 'pose_map_train.csv'
pose_map_val = CSV_DIR / 'pose_map_val.csv'
pose_map_test = CSV_DIR / 'pose_map_test.csv'

# Model hyperparameters
INPUT_SIZE = 126  # 42 keypoints × 3 coords
HIDDEN_SIZE = 256
NUM_LAYERS = 2
MAX_FRAMES = 30
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
NUM_EPOCHS = 10  # Use fewer epochs for interactive training
DROPOUT = 0.3

print(f"Configuration:")
print(f"  Input size: {INPUT_SIZE}")
print(f"  Hidden size: {HIDDEN_SIZE}")
print(f"  Sequence length: {MAX_FRAMES}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"\nPaths:")
print(f"  ML directory: {ML_DIR}")
print(f"  Train CSV: {train_csv}")
print(f"  Val CSV: {val_csv}")
print(f"  Pose dir: {POSE_DIR}")
print(f"  Save dir: {SAVE_DIR}")
print(f"  Log dir: {LOG_DIR}")

## 3. Load Dataset

In [ ]:
# Create data loaders
print("Loading datasets...")

train_loader, val_loader, test_loader, gloss_dict = create_data_loaders(
    train_csv=train_csv,
    val_csv=val_csv,
    test_csv=test_csv if test_csv.exists() else None,
    pose_dir=POSE_DIR,
    pose_map_train=pose_map_train if pose_map_train.exists() else None,
    pose_map_val=pose_map_val if pose_map_val.exists() else None,
    pose_map_test=pose_map_test if pose_map_test.exists() else None,
    batch_size=BATCH_SIZE,
    num_workers=2,  # Fewer workers for notebook
    max_frames=MAX_FRAMES
)

NUM_CLASSES = len(gloss_dict)
idx_to_sign = {v: k for k, v in gloss_dict.items()}

print(f"\nDataset loaded:")
print(f"  Number of classes: {NUM_CLASSES}")
print(f"  Training samples: {len(train_loader.dataset)}")
print(f"  Validation samples: {len(val_loader.dataset)}")
if test_loader:
    print(f"  Test samples: {len(test_loader.dataset)}")

print(f"\nFirst 20 signs:")
for i in range(min(20, NUM_CLASSES)):
    print(f"  {i}: {idx_to_sign[i]}")

## 4. Model Architecture

Bidirectional LSTM following ASL Citizen training approach.

In [ ]:
class ASLLSTMClassifier(nn.Module):
    """Bidirectional LSTM classifier for ASL sign recognition."""
    
    def __init__(self, input_size=126, hidden_size=256, num_classes=100,
                 num_layers=2, dropout=0.3):
        super(ASLLSTMClassifier, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_size * 2, hidden_size)  # *2 for bidirectional
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        # LSTM output
        lstm_out, (h_n, c_n) = self.lstm(x)
        
        # Use final hidden state (concat forward and backward)
        forward_hidden = h_n[-2, :, :]
        backward_hidden = h_n[-1, :, :]
        hidden = torch.cat([forward_hidden, backward_hidden], dim=1)
        
        # Fully connected layers
        out = self.fc1(hidden)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

# Create model
model = ASLLSTMClassifier(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_classes=NUM_CLASSES,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(device)

# Model summary
print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 5. Training Setup

In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Learning rate scheduler (cosine annealing)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=10,
    eta_min=LEARNING_RATE * 0.01
)

# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'lr': []
}

print("Training setup complete!")

## 6. Training Functions

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    
    total_loss = 0.0
    correct = 0
    total = 0
    
    for landmarks, info, labels in tqdm(dataloader, desc="Training"):
        landmarks = landmarks.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(landmarks)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Statistics
        total_loss += loss.item()
        pred = torch.softmax(outputs, dim=1)
        pred_args = torch.argmax(pred, dim=1)
        true_args = torch.argmax(labels, dim=1)
        correct += (pred_args == true_args).sum().item()
        total += labels.shape[0]
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    
    return avg_loss, accuracy


def validate(model, dataloader, criterion, device):
    """Validate on validation set."""
    model.eval()
    
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for landmarks, info, labels in tqdm(dataloader, desc="Validation"):
            landmarks = landmarks.to(device)
            labels = labels.to(device)
            
            # Forward pass
            outputs = model(landmarks)
            loss = criterion(outputs, labels)
            
            # Statistics
            total_loss += loss.item()
            pred = torch.softmax(outputs, dim=1)
            pred_args = torch.argmax(pred, dim=1)
            true_args = torch.argmax(labels, dim=1)
            correct += (pred_args == true_args).sum().item()
            total += labels.shape[0]
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    
    return avg_loss, accuracy

## 7. Train Model

In [ ]:
# Create save directory
SAVE_DIR.mkdir(parents=True, exist_ok=True)

best_val_acc = 0.0

print("Starting training...\n")

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"Epoch {epoch}/{NUM_EPOCHS}")
    print("-" * 50)
    
    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    # Validate
    val_loss, val_acc = validate(
        model, val_loader, criterion, device
    )
    
    # Update learning rate
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    # Print stats
    print(f"LR: {current_lr:.6f}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        
        # Save model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'train_acc': train_acc,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'gloss_dict': gloss_dict
        }, SAVE_DIR / 'best_model.pt')
        
        print(f"✓ Saved best model (val_acc: {val_acc:.4f})")
    
    print()

print("\n" + "="*50)
print("Training Complete!")
print("="*50)
print(f"Best validation accuracy: {best_val_acc:.4f}")

## 8. Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Accuracy', marker='o')
axes[1].plot(history['val_acc'], label='Val Accuracy', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate plot
axes[2].plot(history['lr'], marker='o', color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].grid(True, alpha=0.3)
axes[2].set_yscale('log')

plt.tight_layout()
plt.show()

# Print summary
print(f"Training Summary:")
print(f"  Best Val Accuracy: {max(history['val_acc']):.4f}")
print(f"  Final Train Accuracy: {history['train_acc'][-1]:.4f}")
print(f"  Final Val Accuracy: {history['val_acc'][-1]:.4f}")

## 9. Test Evaluation (Optional)

In [ ]:
if test_loader:
    # Load best model
    checkpoint = torch.load(SAVE_DIR / 'best_model.pt')
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Test
    test_loss, test_acc = validate(model, test_loader, criterion, device)
    
    print(f"Test Results:")
    print(f"  Test Loss: {test_loss:.4f}")
    print(f"  Test Accuracy: {test_acc:.4f}")
else:
    print("Test set not available. Extract test landmarks first.")

## 10. Sample Predictions

In [ ]:
# Get a batch from validation set
model.eval()
landmarks, info, labels = next(iter(val_loader))
landmarks = landmarks.to(device)

with torch.no_grad():
    outputs = model(landmarks)
    probabilities = torch.softmax(outputs, dim=1)
    confidence, predicted = probabilities.max(1)

true_args = torch.argmax(labels, dim=1)

# Display first 10 predictions
print("Sample Predictions:\n")
print(f"{'True Sign':<20} {'Predicted Sign':<20} {'Confidence':<12} {'Correct'}")
print("-" * 70)

for i in range(min(10, len(predicted))):
    true_sign = idx_to_sign[true_args[i].item()]
    pred_sign = idx_to_sign[predicted[i].item()]
    conf = confidence[i].item()
    correct = "✓" if true_sign == pred_sign else "✗"
    
    print(f"{true_sign:<20} {pred_sign:<20} {conf:<12.4f} {correct}")

## 11. Save Model for Backend

In [ ]:
# Save gloss dictionary
gloss_dict_path = SAVE_DIR / 'gloss_dict.json'
with open(gloss_dict_path, 'w') as f:
    json.dump(gloss_dict, f, indent=2)

print(f"Model saved to: {SAVE_DIR / 'best_model.pt'}")
print(f"Gloss dictionary saved to: {gloss_dict_path}")
print(f"\nModel Info:")
print(f"  Number of classes: {NUM_CLASSES}")
print(f"  Input size: {INPUT_SIZE}")
print(f"  Hidden size: {HIDDEN_SIZE}")
print(f"  Sequence length: {MAX_FRAMES}")
print(f"  Best validation accuracy: {best_val_acc:.4f}")

print(f"\nTo copy to backend:")
print(f"  cp {SAVE_DIR / 'best_model.pt'} ../../backend/trained_models/asl_classifier.pth")
print(f"  cp {gloss_dict_path} ../../backend/trained_models/")

---

## Next Steps

### For Full Training (100 epochs, 48-72 hours):
```bash
cd ../scripts
python train_asl_lstm.py --epochs 100 --batch-size 32
```

### For Backend Integration:
1. Copy trained model to backend:
   ```bash
   cp ../trained_models/best_model.pt ../../backend/trained_models/asl_classifier.pth
   ```

2. Update backend service to load the model

3. Test API inference

### For Further Analysis:
- Create confusion matrix for detailed error analysis
- Visualize attention weights (if using attention LSTM)
- Analyze per-class accuracy
- Test on custom video samples

Model ready for deployment! 🚀